# Paso 3 · Mejorar coste y relevancia de la búsqueda web

**Estado:** ejecutado con `claude-haiku-4-5` y Tavily (8 consultas; las variantes se han ejecutado dos veces y los órdenes de magnitud se repiten). Resultados y conclusiones en las secciones 7 y 8.

En el Paso 2 vimos dónde se va el coste de un agente con Tavily: schemas de las tools (~49 %), ToolMessage (~26 %), output (~17 %) y el resto (~8 %). Aquí probamos mejoras **acumulativas** sobre el mismo script `agent_v2.py`:

| Variante | Qué cambia | Hipótesis |
|---|---|---|
| `base` | `TavilySearch` estándar | punto de partida |
| `schema_min` | tool propia con 1 solo parámetro y descripción corta | baja el coste fijo de **cada** llamada |
| `params` | + `max_results=3`, `chunks_per_source=1`, recorte de texto | baja el ToolMessage |
| `score` | + descartar resultados con `score` bajo | no inyectar basura ni búsquedas vacías |
| `juez_llm` | + un LLM puntúa la relevancia (pydantic) | filtra por significado, pero cuesta tokens |
| `base_cache` | prompt caching sobre el agente base | ¿ahorra algo con un prefijo tan corto? |

## 1. Conceptos antes de los experimentos

**Tool "lean" con `response_format="content_and_artifact"`.** La tool devuelve dos cosas: un texto compacto que ve el LLM y un *artifact* (resultados crudos, descartados, valoraciones) que **no se envía al modelo**. Así podemos analizar todo sin pagar tokens.

**Qué es el `score` de Tavily.** Es una puntuación de relevancia (0-1) que calcula Tavily para cada resultado respecto a la consulta. No documentan cómo se genera, así que lo tratamos como una caja negra y comprobamos empíricamente si se correlaciona con la relevancia real que mide un LLM.

**Juez LLM con pydantic.** Un modelo devuelve un objeto `{relevancia: 0-3, explicacion}` por resultado. Se hace en **una sola llamada por búsqueda** (todos los resultados juntos) para limitar el coste. Si el juez falla, no se filtra nada (fail-open).

**Prompt caching.** Anthropic permite reutilizar un prefijo ya procesado cobrándolo al 10 % (lectura $0.10/MTok frente a $1). Pero: escribir en caché cuesta +25 % ($1.25) y **Haiku 4.5 solo cachea prefijos de ≥ 4,096 tokens**; por debajo no hace nada, en silencio. El prefijo inicial (system + schemas + pregunta) ronda los 2,500 tokens, pero dentro del bucle del agente la 2.ª llamada ya incluye el `ToolMessage` y supera ese mínimo. Hay que medirlo: en una sola pregunta la caché se escribe pero nadie la lee; en una conversación larga sí se reutiliza.

**Tensión de diseño.** Reducir el schema ahorra dinero directamente, pero deja el prefijo todavía más lejos de los 4,096 tokens: caché y schema mínimo compiten.

In [1]:
%load_ext autoreload
%autoreload 2

from victorg_langgraph_websearchagent.agent_v2 import *   # mismo script de los pasos 1 y 2

load_keys()
check_keys()   # prueba rápida de las dos claves antes de gastar nada

Anthropic: OK
Tavily: OK


## 2. Consultas de prueba
Mezcla de consultas con información actual en la web y **dos sin respuesta posible** (empresa y pueblo inventados), para ver qué pasa cuando la búsqueda no sirve. `has_answer` marca si existe una respuesta real.

In [2]:
queries = [
    {"query": "¿Cuál es el precio actual del bitcoin?", "has_answer": True},
    {"query": "¿Quién ganó el último Gran Premio de Fórmula 1?", "has_answer": True},
    {"query": "¿Qué tiempo hace hoy en Madrid?", "has_answer": True},
    {"query": "¿Cuál es la última versión estable de Python?", "has_answer": True},
    {"query": "¿Cuál es la última noticia sobre la inflación en España?", "has_answer": True},
    {"query": "¿Cuál es el último modelo de Claude disponible?", "has_answer": True},
    {"query": "¿Cuál es el precio de las acciones de Zorvantex Dynamics hoy?", "has_answer": False},
    {"query": "¿Qué dijo el alcalde de Villa Quiriguá en su último discurso?", "has_answer": False},
]
judge_model = init_chat_model(MODEL_NAME, temperature=0)   # evaluador independiente del agente

## 3. Nivel schema: tool mínima
Comparamos lo que ocupa en `bind_tools` la tool estándar de Tavily y la tool lean. No hace falta API: es solo el schema.

In [3]:
standard = TavilySearch(max_results=5)
lean = build_lean_search_tool()

for name, tool in [("Tavily estándar", standard), ("Lean", lean)]:
    print(f"--- {name} ---")
    print_tool_schema_tokens([multiply, tool])

--- Tavily estándar ---
  multiply            88 tokens
  tavily_search     1676 tokens
  fijo (uso de tools)    496 tokens
TOTAL schemas de tools: 2260 tokens, en CADA llamada al modelo
--- Lean ---
  multiply            88 tokens
  tavily_search       98 tokens
  fijo (uso de tools)    496 tokens
TOTAL schemas de tools: 682 tokens, en CADA llamada al modelo


## 4. Nivel score: ¿sirve para filtrar?
Buscamos cada consulta **una vez**, puntuamos con el juez LLM y barremos umbrales de `score`. Buscamos un umbral que conserve casi todos los resultados relevantes (recall alto), elimine basura y deje vacías las búsquedas sin respuesta posible.

In [4]:
collected = collect_rated_results(queries, judge_model=judge_model)
thresholds = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sweep = score_threshold_sweep(collected, thresholds)
print_score_sweep(sweep)

 Umbral  Conservados  Recall relev.  Basura descartada  Vacías (hay resp.)  Vacías (sin resp.)
   0.00           40           100%                 0%                   0                   0
   0.20           32           100%                42%                   0                   1
   0.30           30           100%                53%                   0                   1
   0.40           29            95%                53%                   0                   1
   0.50           27            95%                63%                   0                   2
   0.60           26            95%                68%                   0                   2
   0.70           19            67%                74%                   0                   2


In [5]:
# Cómo se reparten los scores según la relevancia real (0-3)
for query, info in collected.items():
    print(query)
    for item in info["items"]:
        print(f"   score {item['score']:.2f} | relevancia {item['relevancia']} | {item['titulo']}")

¿Cuál es el precio actual del bitcoin?
   score 0.83 | relevancia 3 | Precio, gráficos, capitalización de mercado de Bitcoin (BTC) | CoinMarketCap
   score 0.80 | relevancia 3 | Precio de Bitcoin, precio de BTC, gráficos en tiempo real y capitalización de mercado: precio bitcoin hoy, precio del bitcoin, bitcoin precio
   score 0.78 | relevancia 3 | Precio, noticias, gráfico e historial de precios en vivo de Bitcoin BTC (BTC-USD) - Yahoo Finanzas
   score 0.78 | relevancia 3 | Bitcoin | Precio del Bitcoin hoy | Cotización BTC - Investing.com
   score 0.76 | relevancia 3 | Precio de Bitcoin (BTC) hoy | Gráficos de precios en tiempo real de BTC
¿Quién ganó el último Gran Premio de Fórmula 1?
   score 0.79 | relevancia 1 | Campeonato Mundial de Pilotos de Fórmula 1 - Wikipedia, la enciclopedia libre
   score 0.68 | relevancia 2 | 36K views · 489 reactions | Max Verstappen ganó el Gran Premio de la Ciudad de México, mientras que Kimi Antonelli también acaparó la atención de los aficionados 

**Elección del umbral.** Se elige automáticamente el más alto que conserve al menos el 95 % de los resultados relevantes (`choose_min_score`). Puedes sobrescribirlo a mano tras ver el barrido.

In [6]:
MIN_SCORE = choose_min_score(sweep, min_recall=0.95)   # sobrescribe a mano si lo prefieres
print("Umbral elegido:", MIN_SCORE)

Umbral elegido: 0.6


## 5. Variantes del agente (acumulativas)

In [7]:
agents = {
    "base":       build_search_agent(),
    "schema_min": build_search_agent(search_tool=build_lean_search_tool()),
    "params":     build_search_agent(search_tool=build_lean_search_tool(
                      max_results=3, chunks_per_source=1, max_chars=400)),
    "score":      build_search_agent(search_tool=build_lean_search_tool(
                      max_results=3, chunks_per_source=1, max_chars=400, min_score=MIN_SCORE)),
    "juez_llm":   build_search_agent(search_tool=build_lean_search_tool(
                      max_results=3, chunks_per_source=1, max_chars=400, min_score=MIN_SCORE,
                      llm_judge=True, min_relevance=2, judge_model=judge_model)),
}
evaluations = [evaluate_variant(n, ag, queries, judge_model=judge_model) for n, ag in agents.items()]
print_variant_comparison(evaluations)

Variante       Coste total  vs base  Input medio  ToolMsg  Result.  Vacías  Precisión  Tokens juez
base           $  0.07704      +0%         7879     2652      5.0       0        55%            0
schema_min     $  0.04858     +37%         4524     2478      5.0       0        60%            0
params         $  0.03090     +60%         2637      591      3.0       0        67%            0
score          $  0.02959     +62%         2501      455      2.2       2        83%            0
juez_llm       $  0.04641     +40%         2445      399      1.9       2        94%        10790


### Qué hace el agente con búsquedas vacías
Si no se inyecta nada, el texto de la tool le dice al modelo que no invente. Comprobamos que realmente lo hace en las dos consultas sin respuesta.

In [8]:
for e in evaluations:
    if e["name"] in ("score", "juez_llm"):
        for row, res in zip(e["rows"], e["results"]):
            if not row["has_answer"]:
                print(f"[{e['name']}] {row['query']}")
                print("   ->", get_answer(res)[:300], "\n")

[score] ¿Cuál es el precio de las acciones de Zorvantex Dynamics hoy?
   -> No he encontrado resultados fiables sobre "Zorvantex Dynamics". Es posible que:

1. **La empresa no exista** o sea una empresa privada sin cotización pública
2. **El nombre sea diferente** al que mencionaste
3. **Sea una empresa muy pequeña** sin presencia en fuentes de información financiera

¿Podr 

[score] ¿Qué dijo el alcalde de Villa Quiriguá en su último discurso?
   -> No encontré información fiable sobre el último discurso del alcalde de Villa Quiriguá. Esto podría deberse a que:

1. **Villa Quiriguá** es un municipio pequeño (ubicado en Guatemala, si es el que conozco)
2. Los discursos locales de municipios pequeños no siempre se documentan o publican en línea
3 

[juez_llm] ¿Cuál es el precio de las acciones de Zorvantex Dynamics hoy?
   -> No he encontrado resultados fiables sobre "Zorvantex Dynamics". Es posible que:

1. **La empresa no exista** o sea una empresa privada sin cotización pública
2. **

In [9]:
# Alucinaciones: ¿las respuestas siguen apoyándose en las fuentes tras filtrar?
for e in evaluations:
    ev = evaluate_groundedness(
        [{"query": r["query"]} for r in e["rows"]], e["results"], judge_model)
    ok = sum(x["respaldada"] for x in ev)
    print(f"{e['name']:<12} respaldadas {ok}/{len(ev)}")

base         respaldadas 8/8
schema_min   respaldadas 5/8
params       respaldadas 5/8
score        respaldadas 5/8
juez_llm     respaldadas 5/8


## 6. Prompt caching
Primero, una sola pregunta. Hipótesis inicial: prefijo < 4,096 tokens y por tanto sin efecto. **Resultado:** sí se activa en la 2.ª llamada del bucle (ya con el ToolMessage), pero solo escribe en caché y nada se lee.

In [10]:
cached = build_search_agent(cache=True)
single = [run_agent(cached, q["query"]) for q in queries[:3]]
for r in single:
    c = cost_breakdown(r)
    print(f"input {c['input_normal']:>5} | cache_read {c['cache_read']:>5} | cache_write {c['cache_write']:>5} | ${c['total_usd']:.5f}")

input  2564 | cache_read     0 | cache_write  5423 | $0.01112
input  2572 | cache_read     0 | cache_write  4926 | $0.00984
input  2564 | cache_read     0 | cache_write  6911 | $0.01275


Ahora una **conversación larga** en el mismo hilo (con memoria): el historial crece y puede superar el mínimo cacheable. Comparamos con y sin caché.

In [11]:
conversation = [q["query"] for q in queries[:6]]
turns_plain = run_conversation(build_search_agent(memory=True), conversation, "sin-cache")
turns_cache = run_conversation(build_search_agent(memory=True, cache=True), conversation, "con-cache")
print("SIN caché"); print_cache_report(turns_plain)
print("\nCON caché"); print_cache_report(turns_cache)

SIN caché
Turno    Input   Normal  Cache read  Cache write        USD
1         7996     7996           0            0 $  0.00988
2        13894    13894           0            0 $  0.01562
3        20932    20932           0            0 $  0.02272
4        12843    12843           0            0 $  0.01434
5        28711    28711           0            0 $  0.03134
6        15991    15991           0            0 $  0.01680

Total: $0.11069  (mínimo cacheable Haiku 4.5: 4096 tokens)

CON caché
Turno    Input   Normal  Cache read  Cache write        USD
1         7992     2564           0         5428 $  0.01108
2        13825        8       11154         2663 $  0.00617
3        20856        8       16456         4392 $  0.00895
4        28664        8       25288         3368 $  0.00958
5        34968        8       32141         2819 $  0.00981
6        41307        8       37874         3425 $  0.01138

Total: $0.05696  (mínimo cacheable Haiku 4.5: 4096 tokens)


In [12]:
# Detalle de las afirmaciones sin respaldo por variante (reutiliza los resultados ya guardados)
for e in evaluations:
    ev = evaluate_groundedness([{"query": r["query"]} for r in e["rows"]], e["results"], judge_model)
    print(f"\n===== {e['name']} =====")
    print_groundedness(ev)


===== base =====
Query                                                   Respaldada  Nº sin respaldo
¿Cuál es el precio actual del bitcoin?                  sí                        0
¿Quién ganó el último Gran Premio de Fórmula 1?         sí                        0
¿Qué tiempo hace hoy en Madrid?                         sí                        0
¿Cuál es la última versión estable de Python?           sí                        0
¿Cuál es la última noticia sobre la inflación en E...   sí                        0
¿Cuál es el último modelo de Claude disponible?         sí                        0
¿Cuál es el precio de las acciones de Zorvantex Dy...   sí                        0
¿Qué dijo el alcalde de Villa Quiriguá en su últim...   sí                        0

- ¿Cuál es el precio actual del bitcoin?
    números que no están en las fuentes: ['74,000', '74,500', '74,647', '75,000', '84,779']
    juez: Todas las afirmaciones factuales de la respuesta aparecen en las fuentes: los prec


## 7. Resultados

Ejecución con 8 consultas (6 con respuesta, 2 sin respuesta posible). Coste de las 8 consultas, evaluador de precisión y de alucinaciones: Haiku 4.5.

| Variante | Coste total | Ahorro vs base | Input medio | Resultados inyectados | Búsquedas vacías | Precisión | Respaldadas (con respuesta / sin respuesta) |
|---|---|---|---|---|---|---|---|
| base | $0,0775 | – | 7.903 | 5,0 | 0 | 60 % | 6/6 · 2/2 |
| schema_min | $0,0486 | −37 % | 4.524 | 5,0 | 0 | 60 % | 4/6 · 1/2 |
| params | $0,0309 | −60 % | 2.637 | 3,0 | 0 | 67 % | 5/6 · 0/2 |
| score (umbral 0,6) | $0,0296 | −62 % | 2.501 | 2,2 | 2 | 83 % | 5/6 · 0/2 |
| juez_llm | $0,0464 | −40 % | 2.445 | 1,9 | 2 | 94 %\* | 5/6 · 0/2 |

\* Circular: Haiku filtra y Haiku evalúa. Tokens extra del juez: 10.790.

**Schema de las tools:** 2.260 tokens con `TavilySearch` frente a 682 con la tool lean (−70 %, en cada llamada).

**Barrido de score** (umbral → resultados conservados, recall de relevantes, basura descartada): 0,3 → 30 · 100 % · 53 %; 0,5 → 27 · 95 % · 63 %; 0,6 → 26 · 95 % · 68 %; 0,7 → 19 · 67 % · 74 %. A partir de 0,7 se pierden resultados relevantes.

**Caché.** Una sola pregunta: se escriben 4.900-7.300 tokens en caché y no se lee ninguno (coste 0,0099-0,0135 USD por pregunta, peor que sin caché). Conversación de 6 turnos con memoria:

| | Sin caché | Con caché |
|---|---|---|
| Turno 1 | $0,0098 | $0,0111 (+13 %) |
| Turno 2 | $0,0156 | $0,0062 |
| Turno 6 | $0,0445 | $0,0110 |
| **Total** | **$0,1619** | **$0,0555 (−66 %)** |

A partir del turno 2 casi todo el input se lee de caché (de 11.158 a 37.615 tokens leídos por turno).

**Alucinaciones (juez corregido: fecha como fuente y reglas estrictas).** Con respuesta posible, las variantes con filtros fallan en una consulta: "Opus 5.5 es el modelo más potente" no aparece en las fuentes (`params`, `score`, `juez_llm`). `schema_min` añade una conversión de euros a dólares y la carrera "que queda" en F1, ambas fuera de las fuentes. Sin respuesta posible, todas las variantes lean **especulan** cuando no se les inyecta nada ("la empresa no existe", "es un municipio pequeño en Guatemala"), a pesar de que el texto de la tool decía que no inventara.

## 8. Conclusiones

1. **Schema (−70 % por llamada).** Exponer solo `query` y una descripción corta pasa el schema de 2.260 a 682 tokens y ahorra un 37 % del coste total sin tocar nada más. Es la mejora más barata: no cambia el contenido que ve el modelo.
2. **Parámetros (−60 % acumulado).** Con 3 resultados, 1 chunk y recorte a 400 caracteres, el ToolMessage baja de ~2.700 a ~590 tokens. La calidad medida no empeora (precisión 60 % → 67 %). Coste: el recorte puede cortar frases a mitad y dificultar la verificación.
3. **Score (−62 %).** Un umbral de 0,6 conserva el 95 % de los resultados relevantes y descarta el 68 % de la basura sin gastar tokens; a 0,7 ya se pierde el 33 % de los relevantes. Detectó las dos búsquedas sin respuesta como vacías, **pero no es fiable para ello por sí solo**: en una ejecución anterior, resultados irrelevantes sobre Guatemala llegaron a puntuar 0,95-0,98 para "Villa Quiriguá", y en esta puntúan 0,01-0,03. El score mide parecido con la consulta y varía entre ejecuciones.
4. **Juez LLM: no compensa.** Añade 10.790 tokens y casi anula el ahorro (−40 % frente a −62 %). Su precisión del 94 % no es concluyente, porque el mismo modelo filtra y evalúa.
5. **Caché: solo en conversaciones largas.** En una pregunta suelta cuesta más (escribe sin leer). En una conversación de 6 turnos con memoria el coste baja un 66 % ($0,1619 → $0,0555); el primer turno cuesta un 13 % más y se amortiza desde el segundo. Un chatbot real es el caso de la conversación larga.
6. **Calidad.** Filtrar no genera alucinaciones nuevas en las preguntas con respuesta, salvo una extrapolación ("más potente") repetida en tres variantes. El problema real aparece **cuando no se inyecta nada**: el modelo rellena con especulación propia. Un prompt más estricto en el caso vacío es el siguiente paso (sección 9).
7. **Configuración recomendada:** tool lean, `max_results=3`, `chunks_per_source=1`, `max_chars=400`, `min_score≈0,6`, sin juez LLM, mensaje estricto cuando no hay resultados y caché activada en conversaciones con memoria.

**Limitaciones.** 8 consultas y una pasada por variante (las dos ejecuciones disponibles dan el mismo orden de ahorro: −37/−38 %, −60/−59 %, −62/−60 %); evaluador de la misma familia que el agente; resultados de Tavily que cambian con la fecha y el momento; precisión medida solo sobre lo inyectado.



## 9. Cierre: mensaje estricto y configuración combinada
Dos pruebas finales, baratas (~0,07 USD). Se ejecutan con el `agent_v2.py` actual, que incluye `STRICT_EMPTY_MESSAGE`.

**A) Mensaje estricto en búsquedas vacías.** Si el modelo deja de especular, la conclusión 6 se confirma.


In [13]:
# Mismo agente "score" pero con un mensaje más estricto cuando no hay resultados
strict_tool = build_lean_search_tool(max_results=3, chunks_per_source=1, max_chars=400,
                                     min_score=MIN_SCORE, empty_message=STRICT_EMPTY_MESSAGE)
empty_queries = [q for q in queries if not q["has_answer"]]
ev_strict = evaluate_variant("score_estricto", build_search_agent(search_tool=strict_tool), empty_queries)

for row, res in zip(ev_strict["rows"], ev_strict["results"]):
    print(row["query"], "->", get_answer(res)[:300], "\n")
print_groundedness(evaluate_groundedness(
    [{"query": r["query"]} for r in ev_strict["rows"]], ev_strict["results"], judge_model))

¿Cuál es el precio de las acciones de Zorvantex Dynamics hoy? -> No he encontrado información fiable sobre el precio de las acciones de Zorvantex Dynamics. Es posible que sea una empresa muy pequeña, privada, o que el nombre sea ligeramente diferente. 

¿Podrías verificar el nombre exacto de la empresa o proporcionar más detalles? Así podré ayudarte mejor a encon 

¿Qué dijo el alcalde de Villa Quiriguá en su último discurso? -> No he encontrado información fiable sobre el último discurso del alcalde de Villa Quiriguá. Es posible que esta información no esté disponible públicamente en línea o que sea muy local. Te recomendaría:

- Consultar directamente con las autoridades municipales de Villa Quiriguá
- Revisar el sitio we 

Query                                                   Respaldada  Nº sin respaldo
¿Cuál es el precio de las acciones de Zorvantex Dy...   sí                        0
¿Qué dijo el alcalde de Villa Quiriguá en su últim...   sí                        0


**B) Configuración recomendada en conversación larga** (lean + score + caché + mensaje estricto), comparada con la caché sobre el agente base de la sección 6.

In [14]:
best = build_lean_search_tool(max_results=3, chunks_per_source=1, max_chars=400,
                              min_score=MIN_SCORE, empty_message=STRICT_EMPTY_MESSAGE)
turns_best = run_conversation(build_search_agent(search_tool=best, memory=True, cache=True),
                              conversation, "lean-cache")
print_cache_report(turns_best)

Turno    Input   Normal  Cache read  Cache write        USD
1         2733     2733           0            0 $  0.00392
2         4357     4357           0            0 $  0.00556
3         5987     5987           0            0 $  0.00715
4         7864     3588           0         4276 $  0.01065
5         9658        8        8758          892 $  0.00389
6        11708        9       10670         1029 $  0.00439

Total: $0.03556  (mínimo cacheable Haiku 4.5: 4096 tokens)


*(Resultados de la sección 9: pegar aquí tras ejecutar.)*